# Lab 03: ChromaDB Basics — SOLUTION

**Goal:** Learn ChromaDB — create collections, add documents with metadata, and run similarity searches.

**What you'll learn:**
- How to create a ChromaDB client and collection
- How to add documents with metadata (source, page, category)
- How to query documents by similarity
- How to filter queries using metadata

## Step 1: Create a ChromaDB client

ChromaDB can run in-memory (for experiments) or persistent (for production).
We'll use in-memory for this lab — fast and no cleanup needed.

We also set up a `SentenceTransformerEmbeddingFunction` so ChromaDB
uses our already-installed `all-MiniLM-L6-v2` model instead of downloading a separate copy.

In [ ]:
import os
from dotenv import load_dotenv
import huggingface_hub

load_dotenv()
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    huggingface_hub.login(token=hf_token, add_to_git_credential=False)
    print("HuggingFace token loaded")
else:
    print("Warning: HF_TOKEN not set in .env")

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

client = chromadb.Client()  # In-memory client
# Reuse the sentence-transformers model already installed (avoids separate ONNX download)
embedding_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
print("ChromaDB client ready!")

## Step 2: Create a collection

A collection is like a table in SQL — it holds related documents.
ChromaDB automatically embeds your text when you add documents.

In [ ]:
collection = client.get_or_create_collection("company_handbook", embedding_function=embedding_fn)
print(f"Collection: '{collection.name}'")

## Step 3: Add documents with metadata

Each document has: text (document), metadata (key-value pairs), and an ID.
Metadata lets you filter search results later.

In [ ]:
collection.add(
    documents=[
        "Annual leave is 24 days per year. Unused leave cannot be carried forward to the next year.",
        "Sick leave is 12 days per year. A medical certificate is required for more than 2 consecutive days.",
        "Employees can work from home up to 3 days per week. Core hours are 10 AM to 4 PM IST.",
        "Travel expenses must be submitted with receipts within 7 days of travel.",
        "Internet reimbursement is Rs 1,500 per month for work-from-home employees.",
        "The Bangalore office is at WeWork Embassy Tech Village, 5th Floor.",
        "The Mumbai office is at Worli Business District, Tower A, 12th Floor.",
        "Our tech stack includes Python (FastAPI), React, PostgreSQL, and AWS.",
        "Laptops are provided by the company and replaced every 3 years.",
        "Maternity leave is 26 weeks as per government regulations.",
    ],
    metadatas=[
        {"category": "leave", "source": "handbook.pdf", "page": 5},
        {"category": "leave", "source": "handbook.pdf", "page": 5},
        {"category": "wfh", "source": "handbook.pdf", "page": 8},
        {"category": "expense", "source": "handbook.pdf", "page": 12},
        {"category": "expense", "source": "handbook.pdf", "page": 12},
        {"category": "office", "source": "handbook.pdf", "page": 15},
        {"category": "office", "source": "handbook.pdf", "page": 15},
        {"category": "tech", "source": "tech-guide.pdf", "page": 3},
        {"category": "tech", "source": "tech-guide.pdf", "page": 7},
        {"category": "leave", "source": "handbook.pdf", "page": 6},
    ],
    ids=[f"doc{i}" for i in range(10)],
)

print(f"Added {collection.count()} documents")

## Step 4: Basic similarity search

Ask a question and ChromaDB finds the most similar documents.
Notice: you don't need to use the exact words from the documents!

In [ ]:
results = collection.query(query_texts=["How many holidays do I get?"], n_results=3)

print("Query: 'How many holidays do I get?'\n")
for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"  [{dist:.4f}] {doc}")
    print(f"     Metadata: {meta}\n")

## Step 5: Search with metadata filter

Combine similarity search with metadata filters for precise results.
"Find expense-related info only"

In [ ]:
results = collection.query(query_texts=["What can I claim?"], n_results=3, where={"category": "expense"})

print("Query: 'What can I claim?' (expense only)\n")
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(f"  - {doc}")
    print(f"    Metadata: {meta}\n")

## Step 6: Search with multiple filters

Use `$and` / `$or` operators for complex filters.

In [ ]:
results = collection.query(query_texts=["What technologies do we use?"], n_results=3, where={"source": "tech-guide.pdf"})

for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(f"  - {doc}")
    print(f"    Metadata: {meta}\n")

## TODO 1: Add more documents and search

Add 3-4 new documents about a topic of your choice.
Suggestions: security policies, meeting rules, dress code.
Then query for them and see if ChromaDB finds them.

In [ ]:
collection.add(
    documents=[
        "All employees must use strong passwords with at least 12 characters.",
        "Two-factor authentication is mandatory for accessing company systems.",
        "Report security incidents to security@unigps.in within 24 hours.",
    ],
    metadatas=[
        {"category": "security", "source": "handbook.pdf", "page": 20},
        {"category": "security", "source": "handbook.pdf", "page": 20},
        {"category": "security", "source": "handbook.pdf", "page": 21},
    ],
    ids=["doc10", "doc11", "doc12"],
)

print("New docs added!")
results = collection.query(query_texts=["What are the password rules?"], n_results=3)
for doc in results["documents"][0]:
    print(f"  Found: {doc}")

## TODO 2: Try the $or filter

Search across multiple categories at once.
Find documents that are either "leave" OR "wfh" related.

In [ ]:
results = collection.query(
    query_texts=["What are my benefits?"],
    n_results=5,
    where={"$or": [{"category": "leave"}, {"category": "wfh"}]},
)

print("Leave OR WFH results:")
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(f"  [{meta['category']}] {doc}")

## Key Takeaways

- ChromaDB stores documents + metadata + embeddings
- Similarity search finds relevant docs even with different words
- Metadata filters let you narrow search by category, source, etc.
- ChromaDB auto-embeds your text (no manual embedding needed)